In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures,StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error,r2_score

p=list(Path(".").rglob("california_housing.csv"))
if not p: raise FileNotFoundError("Extract the practical datasets ZIP beside this notebook.")
df=pd.read_csv(p[0]); X=df[["MedInc"]]; y=df["MedHouseVal"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
degrees=[1,2,3,5,8]; models={}; rows=[]

In [ ]:
for d in degrees:
    m=make_pipeline(PolynomialFeatures(degree=d,include_bias=False),StandardScaler(),LinearRegression())
    m.fit(Xtr,ytr); models[d]=m
    rows.append([d,mean_squared_error(ytr,m.predict(Xtr)),mean_squared_error(yte,m.predict(Xte)),r2_score(yte,m.predict(Xte))])
results=pd.DataFrame(rows,columns=["Degree","Train MSE","Test MSE","Test R2"])
display(results.round(4))

In [ ]:
plt.figure(figsize=(8,5))
sample=X.sample(1500,random_state=42); plt.scatter(sample["MedInc"],y.loc[sample.index],alpha=.15,s=12,label="Data")
line=np.linspace(X.MedInc.min(),X.MedInc.max(),300).reshape(-1,1)
for d in [1,2,3,5]: plt.plot(line.ravel(),models[d].predict(line),label=f"Degree {d}")
plt.xlabel("Median Income"); plt.ylabel("Median House Value"); plt.title("Polynomial Regression"); plt.legend(); plt.grid(alpha=.2); plt.show()
plt.figure(figsize=(7,4)); plt.plot(results.Degree,results["Train MSE"],marker="o",label="Train")
plt.plot(results.Degree,results["Test MSE"],marker="o",label="Test")
plt.xlabel("Polynomial Degree"); plt.ylabel("MSE"); plt.title("Bias-Variance Trade-off"); plt.xticks(degrees); plt.legend(); plt.grid(alpha=.2); plt.show()
print("Practical 4 Summary: Polynomial regression degrees 1, 2, 3, 5 and 8 were compared using median income. Train/test errors illustrate how model complexity affects bias and variance.")